# CS 639 · Stage 0 — Canonical Generation Notebook
**Role B / Role C — Triton Kernel Generation**

> **Do not modify any constant in Section 1.** All config is locked by Role A.  
> Your only job: set `ROLE` to `"B"` or `"C"`, then run all cells top-to-bottom.

---


## Section 0 — Role Assignment

Set this to `"B"` (problems 1–50) or `"C"` (problems 51–100).

In [ ]:
# ── ONLY CELL YOU NEED TO EDIT ────────────────────────────────────────────
ROLE = "B"   # <── change to "C" if you are Role C
# ──────────────────────────────────────────────────────────────────────────

assert ROLE in ("B", "C"), f"ROLE must be 'B' or 'C', got {ROLE!r}"
print(f"Running as Role {ROLE}")


## Section 1 — Locked Configuration (Do Not Modify)

These constants are set by Role A and must be identical across all three generation sessions.


In [ ]:
# ── LOCKED BY ROLE A — DO NOT CHANGE ──────────────────────────────────────
LEVELS    = [1]
MODELS    = ['qwen3_8b_think', 'qwen3_8b_nothink']
RUN_NAMES = ['qwen3_8b_think', 'qwen3_8b_nothink']

PROBLEMS_B = range(1, 51)    # Role B's slice
PROBLEMS_C = range(51, 101)  # Role C's slice
# ──────────────────────────────────────────────────────────────────────────

PROBLEM_IDS = list(PROBLEMS_B if ROLE == "B" else PROBLEMS_C)

print(f"Levels      : {LEVELS}")
print(f"Models      : {MODELS}")
print(f"Problem IDs : {PROBLEM_IDS[0]}–{PROBLEM_IDS[-1]} ({len(PROBLEM_IDS)} problems)")


## Section 2 — Environment Check

Verify you are on an A100 40 GB GPU. If not, go to **Runtime → Change runtime type → A100**.


In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected! Switch runtime to GPU (A100) in Colab."
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Device   : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Section 3 — Install Dependencies

This installs KernelBench, vLLM, and helper libraries.  
**Expected time:** ~5–10 minutes on a fresh Colab session.


In [ ]:
import os
from pathlib import Path

if not os.path.isdir("KernelBench"):
    !git clone https://github.com/ScalingIntelligence/KernelBench.git
else:
    print("KernelBench already cloned — skipping.")


In [ ]:
kernelbench_candidates = [Path("KernelBench"), Path("../KernelBench")]
KERNELBENCH_ROOT = next(
    (p for p in kernelbench_candidates
     if (p / "scripts" / "eval_from_generations.py").exists()
     and (p / "src" / "kernelbench").exists()),
    None,
)
if KERNELBENCH_ROOT is None:
    raise FileNotFoundError(
        "Could not find KernelBench checkout. Clone it at ./KernelBench or ../KernelBench."
    )

!pip install -q -e {KERNELBENCH_ROOT}
!pip install -q vllm
!pip install -q tqdm pandas

print(f"KernelBench root: {KERNELBENCH_ROOT}")


In [ ]:
import sys

src_path = str(KERNELBENCH_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import triton
from vllm import LLM, SamplingParams
import kernelbench

print(f"Triton      : {triton.__version__}")
print(f"vLLM        : imported OK")
print(f"KernelBench : imported OK")


## Section 4 — Write env_info.json

Role A requires an `env_info.json` in every run directory. This cell captures the exact runtime
and writes it to both run directories (`qwen3_8b_think/` and `qwen3_8b_nothink/`).

**Run this before generating any kernels.**


In [ ]:
import json, subprocess, importlib
from datetime import datetime, timezone

RUNS_DIR = Path("runs")

def get_version(pkg):
    try:
        return importlib.import_module(pkg).__version__
    except Exception:
        return "not installed"

def get_vllm_version():
    try:
        import vllm
        return vllm.__version__
    except Exception:
        return "not installed"

def get_transformers_version():
    try:
        import transformers
        return transformers.__version__
    except Exception:
        return "not installed"

def get_notebook_runtime():
    """Return a Colab session ID if available, else a timestamp."""
    try:
        import google.colab
        result = subprocess.run(
            ["cat", "/proc/1/cgroup"], capture_output=True, text=True
        )
        for line in result.stdout.splitlines():
            if "docker" in line or "lxc" in line:
                container_id = line.strip().split("/")[-1][:12]
                return f"colab-{container_id}"
    except Exception:
        pass
    return f"local-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

env_info = {
    "role":                ROLE,
    "problem_ids":         f"{PROBLEM_IDS[0]}-{PROBLEM_IDS[-1]}",
    "gpu_model":           torch.cuda.get_device_name(0),
    "cuda_version":        torch.version.cuda,
    "torch_version":       torch.__version__,
    "triton_version":      triton.__version__,
    "transformers_version": get_transformers_version(),
    "vllm_version":        get_vllm_version(),
    "notebook_runtime":    get_notebook_runtime(),
    "recorded_at":         datetime.now(timezone.utc).isoformat(),
}

for run_name in RUN_NAMES:
    run_dir = RUNS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "env_info.json", "w") as f:
        json.dump(env_info, f, indent=2)

print("env_info.json written to:")
for run_name in RUN_NAMES:
    print(f"  runs/{run_name}/env_info.json")
print()
print(json.dumps(env_info, indent=2))


## Section 5 — Model Configuration

Qwen3-8B is loaded **once** and reused for both thinking and non-thinking runs.
Sampling parameters follow the model card recommendations.


In [ ]:
# Sampling parameters — locked by Role A
# Qwen3-8B-think: non-greedy required (model card warning against temp=0 in think mode)
# Qwen3-8B-nothink: slightly higher temperature, thinking disabled

MODEL_CONFIGS = {
    "qwen3_8b_think": {
        "model_id":   "Qwen/Qwen3-8B",
        "run_name":   "qwen3_8b_think",
        "load_kwargs": {"dtype": "bfloat16", "gpu_memory_utilization": 0.85},
        "sampling_params": SamplingParams(
            temperature=0.6, top_p=0.95, top_k=20, max_tokens=4096, seed=0
        ),
        "thinking": True,
        "chat_template_kwargs": {"enable_thinking": True},
    },
    "qwen3_8b_nothink": {
        "model_id":   "Qwen/Qwen3-8B",
        "run_name":   "qwen3_8b_nothink",
        "load_kwargs": {"dtype": "bfloat16", "gpu_memory_utilization": 0.85},
        "sampling_params": SamplingParams(
            temperature=0.7, top_p=0.8, top_k=20, max_tokens=4096, seed=0
        ),
        "thinking": False,
        "chat_template_kwargs": {"enable_thinking": False},
    },
}

print("Model configs:")
for tag, cfg in MODEL_CONFIGS.items():
    think_str = "thinking=ON " if cfg["thinking"] else "thinking=OFF"
    sp = cfg["sampling_params"]
    print(f"  [{tag}]  {think_str}  model={cfg['model_id']}")
    print(f"           temp={sp.temperature}  top_p={sp.top_p}  top_k={sp.top_k}  max_tokens={sp.max_tokens}")


## Section 6 — Helper Functions

In [ ]:
import re
import gc
import csv
import time
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm

from kernelbench.utils import extract_first_code
from kernelbench.prompt_constructor_toml import get_prompt_for_backend
from kernelbench.dataset import construct_kernelbench_dataset


# ── Think-block stripper ───────────────────────────────────────────────────
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def strip_think_block(text: str) -> str:
    """Remove <think>…</think> blocks emitted by Qwen3-8B thinking mode."""
    return _THINK_RE.sub("", text).strip()


# ── Triton validity check ─────────────────────────────────────────────────
def check_triton_validity(source: str) -> dict:
    """
    Lightweight static check on a generated kernel file.
    Returns:
        has_triton_jit      – at least one @triton.jit decorated function
        torch_calls_in_jit  – torch.* calls found inside a @triton.jit block
        is_valid            – has_triton_jit AND NOT torch_calls_in_jit
        needs_manual_review – torch.* found inside @triton.jit (potential hacking)
    """
    has_triton_jit = bool(re.search(r"@triton\.jit", source))
    # Rough heuristic: find torch. usage after a @triton.jit line
    torch_in_jit = False
    if has_triton_jit:
        in_jit = False
        for line in source.splitlines():
            if re.search(r"@triton\.jit", line):
                in_jit = True
            if in_jit and re.search(r"torch\.", line):
                torch_in_jit = True
                break
    return {
        "has_triton_jit":     has_triton_jit,
        "torch_calls_in_jit": torch_in_jit,
        "is_valid":           has_triton_jit and not torch_in_jit,
        "needs_manual_review": torch_in_jit,
    }


# ── vLLM model loader / unloader ──────────────────────────────────────────
def load_vllm_model(model_tag: str) -> LLM:
    cfg = MODEL_CONFIGS[model_tag]
    print(f"Loading {cfg['model_id']} ...")
    llm = LLM(model=cfg["model_id"], **cfg["load_kwargs"])
    print("  Model loaded.")
    return llm

def unload_vllm_model(llm: LLM) -> None:
    del llm
    gc.collect()
    torch.cuda.empty_cache()
    print("  Model unloaded, GPU memory freed.")


# ── CSV helpers ───────────────────────────────────────────────────────────
MANIFEST_COLS = [
    "problem_id", "run_name", "status", "raw_path", "clean_path",
    "kernel_path", "meta_path", "extraction_success", "output_length", "error_message",
]
VALIDITY_COLS = [
    "problem_id", "run_name", "has_triton_jit", "torch_calls_in_jit",
    "is_valid", "needs_manual_review", "notes",
]

def _append_csv(path: Path, cols: list, row: dict) -> None:
    write_header = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if write_header:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in cols})

print("Helper functions defined.")


## Section 7 — Kernel Generation

This is the main generation loop. It:
1. Loads Qwen3-8B once.
2. Generates kernels for **all problems in your assigned range**, for both run names.
3. Saves four files per (problem, run_name): `raw.txt`, `clean.txt`, `kernel.py`, `meta.json`.
4. Appends a row to `generation_manifest.csv` and `validity_flags.csv` after each problem.
5. Skips problems that already have a `kernel.py` (resume-friendly — safe to re-run after interruption).

**Expected time:** ~5–15 min generation + overhead. Watch the progress bar.


In [ ]:
def generate_for_role(llm: LLM) -> None:
    """
    Generate kernels for PROBLEM_IDS across both run names.
    Both run names share the same model weights (Qwen3-8B),
    so we loop over model_tags while the model is in memory.
    """
    for model_tag in MODELS:
        cfg      = MODEL_CONFIGS[model_tag]
        run_name = cfg["run_name"]
        run_dir  = RUNS_DIR / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        manifest_path = run_dir / "generation_manifest.csv"
        validity_path = run_dir / "validity_flags.csv"

        sp          = cfg["sampling_params"]
        thinking    = cfg["thinking"]
        chat_kwargs = cfg.get("chat_template_kwargs")

        dataset = construct_kernelbench_dataset(level=1, source="huggingface")

        print(f"\n{'='*65}")
        print(f"  [{model_tag}]  problems {PROBLEM_IDS[0]}–{PROBLEM_IDS[-1]}")
        print(f"{'='*65}")

        for pid in tqdm(PROBLEM_IDS, desc=model_tag):
            # File paths for this (pid, run_name)
            kernel_path  = run_dir / f"level_1_problem_{pid}_sample_0_kernel.py"
            raw_path     = run_dir / f"level_1_problem_{pid}_raw.txt"
            clean_path   = run_dir / f"level_1_problem_{pid}_clean.txt"
            meta_path    = run_dir / f"level_1_problem_{pid}_meta.json"

            # Resume-friendly: skip if kernel already written
            if kernel_path.exists():
                continue

            t_start = time.time()
            error_msg = ""
            raw_text  = ""
            code      = ""
            extraction_success = False

            try:
                problem = dataset.get_problem_by_id(pid)
                prompt  = get_prompt_for_backend(
                    ref_arch_src=problem.code, backend="triton", option="one_shot"
                )

                messages = [{"role": "user", "content": prompt}]
                if chat_kwargs:
                    outputs = llm.chat(
                        [messages], sampling_params=sp,
                        chat_template_kwargs=chat_kwargs
                    )
                else:
                    outputs = llm.chat([messages], sampling_params=sp)

                raw_text = outputs[0].outputs[0].text

                # Strip <think> blocks for thinking mode
                clean_text = strip_think_block(raw_text) if thinking else raw_text

                # Extract Python code block
                code = extract_first_code(clean_text, ["python", "py"])
                extraction_success = bool(code)
                if not code:
                    code = clean_text  # fall back to full output

            except Exception as e:
                error_msg = str(e)
                clean_text = raw_text

            elapsed = time.time() - t_start

            # ── Write the four required files ───────────────────────────
            raw_path.write_text(raw_text, encoding="utf-8")
            clean_path.write_text(clean_text if 'clean_text' in dir() else "", encoding="utf-8")
            kernel_path.write_text(code, encoding="utf-8")

            meta = {
                "problem_id":         pid,
                "run_name":           run_name,
                "model_tag":          model_tag,
                "model_id":           cfg["model_id"],
                "thinking":           thinking,
                "level":              1,
                "extraction_success": extraction_success,
                "output_length":      len(raw_text),
                "elapsed_seconds":    round(elapsed, 2),
                "error_message":      error_msg,
                "generated_at":       datetime.now(timezone.utc).isoformat(),
            }
            meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

            # ── Validity check ───────────────────────────────────────────
            if extraction_success:
                validity = check_triton_validity(code)
            else:
                validity = {
                    "has_triton_jit":      False,
                    "torch_calls_in_jit":  False,
                    "is_valid":            False,
                    "needs_manual_review": False,
                }

            # ── Append to CSVs ───────────────────────────────────────────
            _append_csv(manifest_path, MANIFEST_COLS, {
                "problem_id":         pid,
                "run_name":           run_name,
                "status":             "ok" if not error_msg else "error",
                "raw_path":           str(raw_path),
                "clean_path":         str(clean_path),
                "kernel_path":        str(kernel_path),
                "meta_path":          str(meta_path),
                "extraction_success": extraction_success,
                "output_length":      len(raw_text),
                "error_message":      error_msg,
            })
            _append_csv(validity_path, VALIDITY_COLS, {
                "problem_id":          pid,
                "run_name":            run_name,
                "has_triton_jit":      validity["has_triton_jit"],
                "torch_calls_in_jit":  validity["torch_calls_in_jit"],
                "is_valid":            validity["is_valid"],
                "needs_manual_review": validity["needs_manual_review"],
                "notes":               error_msg,
            })

        print(f"  [{model_tag}] done. Files in: {run_dir}/")


# ── Load model once, generate for both run names, unload ─────────────────
print("Loading Qwen3-8B (shared weights for both run names) ...")
llm = load_vllm_model("qwen3_8b_think")   # either tag — same model_id
generate_for_role(llm)
unload_vllm_model(llm)

print("\n✓ Generation complete.")


## Section 8 — Verify Outputs

Run this after generation to confirm all expected files exist and CSVs have the right row count.
Fix any reported issues before uploading to Role A.


In [ ]:
import pandas as pd

REQUIRED_FILES = ["raw.txt", "clean.txt", "kernel.py", "meta.json"]

print(f"{'='*60}")
print(f"  Verification for Role {ROLE} — problems {PROBLEM_IDS[0]}–{PROBLEM_IDS[-1]}")
print(f"{'='*60}\n")

all_ok = True

for run_name in RUN_NAMES:
    run_dir = RUNS_DIR / run_name
    print(f"[{run_name}]")

    missing = []
    empty   = []

    for pid in PROBLEM_IDS:
        for suffix, label in [
            ("_sample_0_kernel.py", "kernel.py"),
            ("_raw.txt",            "raw.txt"),
            ("_clean.txt",          "clean.txt"),
            ("_meta.json",          "meta.json"),
        ]:
            fpath = run_dir / f"level_1_problem_{pid}{suffix}"
            if not fpath.exists():
                missing.append(f"  problem {pid}: MISSING {label}")
            elif fpath.stat().st_size == 0:
                empty.append(f"  problem {pid}: EMPTY   {label}")

    for msg in missing + empty:
        print(msg)
        all_ok = False

    # CSV checks
    for csv_name, expected_rows in [
        ("generation_manifest.csv", len(PROBLEM_IDS)),
        ("validity_flags.csv",      len(PROBLEM_IDS)),
    ]:
        csv_path = run_dir / csv_name
        if not csv_path.exists():
            print(f"  MISSING: {csv_name}")
            all_ok = False
        else:
            df = pd.read_csv(csv_path)
            status = "OK" if len(df) == expected_rows else f"WARNING — expected {expected_rows}, got {len(df)}"
            print(f"  {csv_name}: {len(df)} rows  [{status}]")
            if len(df) != expected_rows:
                all_ok = False

    # Validity summary
    vf_path = run_dir / "validity_flags.csv"
    if vf_path.exists():
        vf = pd.read_csv(vf_path)
        n_valid   = vf["is_valid"].sum()
        n_review  = vf["needs_manual_review"].sum()
        print(f"  Valid kernels : {n_valid}/{len(vf)}")
        print(f"  Manual review : {n_review}/{len(vf)}")

    print()

if all_ok:
    print("✓ All files present and non-empty. Ready to upload to Role A.")
else:
    print("✗ Some issues found — fix before uploading. Re-run Section 7 to fill gaps.")


## Section 9 — Package for Upload

Creates a single zip archive (`role_{B|C}_shard.zip`) containing:
- Both run directories with all generated files
- Both CSVs per run directory
- `env_info.json` for both run directories

Send this zip to Role A.


In [ ]:
import zipfile

zip_name = f"role_{ROLE}_shard.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for run_name in RUN_NAMES:
        run_dir = RUNS_DIR / run_name
        if not run_dir.exists():
            print(f"WARNING: {run_dir} does not exist — skipping.")
            continue
        for fpath in sorted(run_dir.iterdir()):
            zf.write(fpath, arcname=str(fpath))

print(f"✓ Created {zip_name}")
print(f"  Contents:")

with zipfile.ZipFile(zip_name, "r") as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"    {name}  ({info.file_size:,} bytes)")


In [ ]:
# Download the zip (Colab only)
try:
    from google.colab import files
    files.download(zip_name)
    print(f"Download triggered for {zip_name}")
except ImportError:
    print(f"Not in Colab. Find your archive at: {zip_name}")
